# D12 独立练习

使用 Python (D Learning) 内核；Shift+Enter 运行，Ctrl+S 保存。
每次先预测行数再运行。参考代码在 reference.ipynb；第 7 节应关掉示例独立完成。
价格样本来自 D:/Learning/study/d12/data/prices.csv。


## 1. 读取与检查

读取 D12 价格，转换数值和日期，查看 shape、info、缺失数。预计 8 行 5 列、空价格 1 个。


In [6]:
from pathlib import Path  # Path：表示和拼接文件路径
import pandas as pd  # pandas 表格处理库，通常简称 pd

base_dir = Path("D:/Learning/study/d12")  # D12 文件夹的绝对路径
prices = pd.read_csv(base_dir / "data" / "prices.csv")  # 读取八行价格记录
prices["price"] = pd.to_numeric(prices["price"], errors="coerce")  # 转换数值，失败变缺失
prices["date"] = pd.to_datetime(  # 转换日期类型
    prices["date"],  # 读取原始日期列
    format="%Y-%m-%d",  # 按 年-月-日 解析
    errors="coerce",  # 非法日期变成 NaT
)  # 转换结果放回 date 列

print(prices)  # 查看原始八条记录
print(prices.shape)  # 预计 (8, 5)
prices.info()  # 查看列类型和非空数量
print(prices.isna().sum())  # 预计 price 缺失 1 个，其他列为 0


   record_id       date asset  price  volume
0        101 2026-10-12     A  100.0    1000
1        102 2026-10-12     B  200.0     800
2        103 2026-10-13     A  105.0    1200
3        104 2026-10-13     B  198.0     900
4        105 2026-10-14     A    NaN    1100
5        106 2026-10-14     B  205.0     850
6        107 2026-10-15     A  108.0    1300
7        108 2026-10-15     B  203.0     950
(8, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   record_id  8 non-null      int64         
 1   date       8 non-null      datetime64[ns]
 2   asset      8 non-null      object        
 3   price      7 non-null      float64       
 4   volume     8 non-null      int64         
dtypes: datetime64[ns](1), float64(1), int64(2), object(1)
memory usage: 448.0+ bytes
record_id    0
date         0
asset        0
price        1
volume       

## 2. 建立映射表

独立创建 asset、asset_name 两列，资产 A/B/C 分别对应资产A/B/C。核对代码唯一，形状 (3,2)。


In [13]:
# 预计映射形状：
# 用 pd.DataFrame 建立映射表。
asset_names = pd.DataFrame(
    {"asset":["A","B","C"],
     "asset_name" :  ["资产A", "资产B", "资产C"],    
    }
)
print(asset_names.shape)
print(asset_names)

(3, 2)
  asset asset_name
0     A        资产A
1     B        资产B
2     C        资产C


## 3. 正常左连接

按 asset 左连接，使用 indicator=True 和 validate='many_to_one'。先预测行数与形状，再查看 _merge。解释为什么结果没有 C。


In [17]:
# 预计行数和形状：
# 在这里写正常 merge 代码。
# 为什么没有 C：
named_prices = pd.merge(
    prices,
    asset_names,
    on = "asset",
    how = "left",
    validate = "many_to_one",
    indicator = True,
)
print(named_prices)
print(named_prices.shape)

   record_id       date asset  price  volume asset_name _merge
0        101 2026-10-12     A  100.0    1000        资产A   both
1        102 2026-10-12     B  200.0     800        资产B   both
2        103 2026-10-13     A  105.0    1200        资产A   both
3        104 2026-10-13     B  198.0     900        资产B   both
4        105 2026-10-14     A    NaN    1100        资产A   both
5        106 2026-10-14     B  205.0     850        资产B   both
6        107 2026-10-15     A  108.0    1300        资产A   both
7        108 2026-10-15     B  203.0     950        资产B   both
(8, 7)


## 4. 未匹配实验

从映射副本中去掉 B，再左连接。保留原映射，使用 _merge=='left_only' 找出未匹配行。预计结果仍 8 行，其中 4 行未匹配。


In [20]:
# 预计结果行数和未匹配行数：
# 在这里构造缺少 B 的映射并合并。
# 名称缺失意味着什么：
names_without_b = asset_names.loc[asset_names["asset"] != "B"].copy()  # 临时只留下 A、C 映射
new_left = pd.merge(
    prices,
    names_without_b,
    on = "asset",
    how = "left",
    validate = "many_to_one",
    indicator = True,
)
unmatched_mask = new_left["_merge"] == "left_only"  # 标记未找到右侧映射的行
print(new_left.loc[unmatched_mask])  # 应看到 B 的四条记录
print("结果行数：", len(new_left))  # 仍然是 8
print("未匹配行数：", int(unmatched_mask.sum()))  # 预计 4

   record_id       date asset  price  volume asset_name     _merge
1        102 2026-10-12     B  200.0     800        NaN  left_only
3        104 2026-10-13     B  198.0     900        NaN  left_only
5        106 2026-10-14     B  205.0     850        NaN  left_only
7        108 2026-10-15     B  203.0     950        NaN  left_only
结果行数： 8
未匹配行数： 4


## 5. 重复映射实验

完整复制第一条 A 映射，concat 后观察 A/B/C/A。暂不加 validate 进行左连接，检查编号 101 是否出现两次。先手算行数。


In [34]:
# A 组匹配计算：
# 总行数预测：
# 在这里写重复映射和 merge。
again_a = asset_names.iloc[[0]].copy()
new_names = pd.concat(
    [asset_names , again_a],
    ignore_index = True,
)
print(new_names)


provided_duplicate = pd.read_csv(base_dir / "data" / "asset_names_duplicate.csv")  # 读取准备的重复样本
print(new_names.equals(provided_duplicate))  # 预计 True

expanded_prices = pd.merge(  # 本次先观察右表重复键的实际匹配结果
    prices,  # 左侧 A 四行、B 四行
    duplicate_names,  # 右侧 A 两行、B 一行、C 一行
    on="asset",  # 仍按资产代码匹配
    how="left",  # 左连接也可能因为右侧重复而增加行数
)  # 不加 validate，仅用于这次重复映射实验
print(expanded_prices)
print("合并前后行数：", len(prices), len(expanded_prices))  # 预计 8、12
print(expanded_prices.groupby("asset").size())  # A 为 8，B 为 4
print(expanded_prices.loc[expanded_prices["record_id"] == 101])  # 编号 101 出现两次


  asset asset_name
0     A        资产A
1     B        资产B
2     C        资产C
3     A        资产A
True
    record_id       date asset  price  volume asset_name
0         101 2026-10-12     A  100.0    1000        资产A
1         101 2026-10-12     A  100.0    1000        资产A
2         102 2026-10-12     B  200.0     800        资产B
3         103 2026-10-13     A  105.0    1200        资产A
4         103 2026-10-13     A  105.0    1200        资产A
5         104 2026-10-13     B  198.0     900        资产B
6         105 2026-10-14     A    NaN    1100        资产A
7         105 2026-10-14     A    NaN    1100        资产A
8         106 2026-10-14     B  205.0     850        资产B
9         107 2026-10-15     A  108.0    1300        资产A
10        107 2026-10-15     A  108.0    1300        资产A
11        108 2026-10-15     B  203.0     950        资产B
合并前后行数： 8 12
asset
A    8
B    4
dtype: int64
   record_id       date asset  price  volume asset_name
0        101 2026-10-12     A  100.0    1000        资产A
1

## 6. 验证并修复

给重复映射实验加 many_to_one，用 try/except 显示预期 MergeError；仅移除代码和名称都相同的副本，再验证恢复八行。


In [36]:
# 在这里观察并处理预期的验证错误。
# 修复后再次合并，不要只凭行数判断成功。
try:  # 尝试验证这份故意有重复键的映射
    pd.merge(  # 右表 A 代码重复，因此合并验证将失败
        prices,  # 左表：价格记录
        duplicate_names,  # 右表：有重复 A 的映射
        on="asset",  # 连接键为 asset
        how="left",  # 使用左连接
        validate="many_to_one",  # 要求右侧 asset 唯一
    )  # 预计抛出 MergeError，转到下面的 except
except pd.errors.MergeError:  # 只处理本节预期的合并关系错误
    print("预期检查：右侧映射键重复，需要先核实和处理。")  # 输出原因，未生成合格的合并结果


fixed_names = duplicate_names.drop_duplicates(  # 只移除内容完全相同的映射副本
    subset=["asset", "asset_name"],  # 代码和名称都相同，才视为完全重复
    keep="first",  # 相同内容保留第一次出现的那条
).copy()  # 用副本继续操作

fixed_prices = pd.merge(  # 修复之后再次验证合并
    prices,  # 原来的八条价格记录
    fixed_names,  # 已移除完全相同副本的映射
    on="asset",  # 根据资产代码匹配
    how="left",  # 保留价格记录
    validate="many_to_one",  # 重新检查右表键唯一，不只看行数
)  # 本例检查通过，恢复八行
print("修复后映射形状：", fixed_names.shape)  # 预计 (3, 2)
print("修复后明细行数：", len(fixed_prices))  # 预计 8
print(fixed_prices)

预期检查：右侧映射键重复，需要先核实和处理。
修复后映射形状： (3, 2)
修复后明细行数： 8
   record_id       date asset  price  volume asset_name
0        101 2026-10-12     A  100.0    1000        资产A
1        102 2026-10-12     B  200.0     800        资产B
2        103 2026-10-13     A  105.0    1200        资产A
3        104 2026-10-13     B  198.0     900        资产B
4        105 2026-10-14     A    NaN    1100        资产A
5        106 2026-10-14     B  205.0     850        资产B
6        107 2026-10-15     A  108.0    1300        资产A
7        108 2026-10-15     B  203.0     950        资产B


## 7. 独立检查：关闭参考，从空白完成六步

关闭 reference.ipynb 和 README 示例。重新读 prices.csv 与正常 asset_names.csv；检查类型、缺失、映射键；筛选 A/B、价格非空且>=100；agg 六项统计；左连接名称并验证；保存 outputs/asset_summary_with_names.csv。每一步写预测行数和实际行数。预计 8→7→2→2，最终 (2,8)。


In [38]:
prices_check = pd.read_csv(base_dir / "data" / "prices.csv")  # 从文件重新读取价格，不依赖前面实验的变量
names_check = pd.read_csv(base_dir / "data" / "asset_names.csv")  # 读取正常、键唯一的名称映射表

prices_check["price"] = pd.to_numeric(  # 将价格转换为数值
    prices_check["price"],  # 需要转换的价格列
    errors="coerce",  # 无法转换的价格变成缺失值
)  # 转换结果写回价格列
prices_check["date"] = pd.to_datetime(  # 将日期转换为日期类型
    prices_check["date"],  # 需要转换的日期列
    format="%Y-%m-%d",  # 按 年-月-日 解析
    errors="coerce",  # 非法日期变成 NaT
)  # 转换结果写回日期列

print("读取后形状：", prices_check.shape)  # 预计 (8, 5)
print(prices_check.isna().sum())  # 价格缺失 1 个，其他列缺失 0 个
print("映射键是否重复：", names_check["asset"].duplicated().any())  # 预计 False

selection = (  # selection：记录每一行是否符合本次筛选要求的布尔 Series
    prices_check["asset"].isin(["A", "B"])  # 资产必须是 A 或 B；isin 表示是否属于给定列表
    & prices_check["price"].notna()  # 价格必须非空；notna 表示不是缺失值
    & (prices_check["price"] >= 100)  # 价格至少为 100；组合条件时给比较式加括号
)  # 三个条件都成立时，整行标记才为 True

filtered = prices_check.loc[selection].copy()  # 保留满足条件的记录，并创建独立副本

print("筛选后形状：", filtered.shape)  # 预计 (7, 5)，编号 105 被排除


summary = filtered.groupby(  # 在筛选结果上按资产分组
    "asset",  # 分组键为资产代码
    as_index=False,  # 让 asset 仍是普通列，方便随后 merge
).agg(  # 一次计算六项统计，并给结果列命名
    record_count=("price", "size"),  # 筛选后每组记录数：A 为 3，B 为 4
    valid_price_count=("price", "count"),  # 非空价格数：A 为 3，B 为 4
    start_date=("date", "min"),  # 每组最早日期
    end_date=("date", "max"),  # 每组最晚日期
    highest_price=("price", "max"),  # 每组最高价
    lowest_price=("price", "min"),  # 每组最低价
)  # 汇总后预计两行七列

##prices----check==filtered----agg====summary-----merge====final_suumary
final_summary = pd.merge(  # 给汇总表补充资产名称
    summary,  # 左表：按资产汇总的两行
    names_check,  # 右表：资产代码与名称映射
    on="asset",  # 按共同的 asset 列匹配
    how="left",  # 保留左表中 A、B 的汇总结果
    validate="many_to_one",  # 检查右表每个资产代码最多一行
)  # 结果预计两行八列，新增一列 asset_name
print(final_summary)  # 查看最终汇总表
print("合并前后行数：", len(summary), len(final_summary))  # 预计 2、2


output_dir = base_dir / "outputs"  # 设置输出文件夹
output_dir.mkdir(parents=True, exist_ok=True)  # 上级目录也可创建；已存在时继续
report_path = output_dir / "asset_summary_with_names.csv"  # 拼接报告文件路径
final_summary.to_csv(  # 导出本次独立流程的最终结果
    report_path,  # 输出到指定报告文件
    index=False,  # 不保存额外的默认行索引
    encoding="utf-8-sig",  # 使用便于 Windows 软件识别的 UTF-8 编码
    date_format="%Y-%m-%d",  # 日期输出为 年-月-日
)  # 报告保存完成
print("最终报告：", report_path)  # 显示结果文件的位置

读取后形状： (8, 5)
record_id    0
date         0
asset        0
price        1
volume       0
dtype: int64
映射键是否重复： False
筛选后形状： (7, 5)
  asset  record_count  valid_price_count start_date   end_date  highest_price  \
0     A             3                  3 2026-10-12 2026-10-15          108.0   
1     B             4                  4 2026-10-12 2026-10-15          205.0   

   lowest_price asset_name  
0         100.0        资产A  
1         198.0        资产B  
合并前后行数： 2 2
最终报告： D:\Learning\study\d12\outputs\asset_summary_with_names.csv


## 8. 保存正常明细

保存第 3 节正常左连接的八行明细到 outputs/prices_with_names.csv。移除 _merge 辅助列，预计 (8,6)，缺失价格仍保留。


In [39]:
output_dir = base_dir / "outputs"  # 指定结果文件夹
output_dir.mkdir(parents=True, exist_ok=True)  # 必要时创建目录，已存在则继续
detail_for_export = named_prices.drop(columns=["_merge"])  # 去掉检查辅助列，不改 named_prices
detail_path = output_dir / "prices_with_names.csv"  # 拼接正常左连接明细的文件路径
detail_for_export.to_csv(  # 保存八行、六列的明细表
    detail_path,  # 输出路径
    index=False,  # 不保存默认数字索引
    encoding="utf-8-sig",  # 使用便于 Windows 识别的编码
    date_format="%Y-%m-%d",  # 日期按 年-月-日 写出
)  # 明细保存完成
print("正常左连接明细：", detail_path)  # 显示文件位置
print(detail_for_export.shape)  # 预计 (8, 6)


正常左连接明细： D:\Learning\study\d12\outputs\prices_with_names.csv
(8, 6)


## 9. 两数之和：独立写函数

定义 two_sum(nums,target)，返回两个不同下标。先用字典保存之前数值的下标，先查再存。重点手算 [5,5]、目标 10。


In [ ]:
# 第一轮：index____，number____，needed____，seen____
# 第二轮：index____，number____，needed____，seen____
# 在这里独立定义函数。


## 10. 检查两数之和

检查 [5,5]→目标10、[3,2,4]→目标6、[2,7,11,15]→目标9、[0,4,3,0]→目标0、[-3,4,3,90]→目标0。选做本地扩展：[3]→目标6 应无解，不能返回 [0,0]。


In [ ]:
# 写下各组预期下标，再调用自己的函数检查。
# 为什么返回下标而不是数值：
# 为什么不能先把当前数字存入 seen 再检查：
